# 📦 Notebook 2 — Parent Chunk Verification

Inspect every parent document loaded into the docstore.
Check chunk counts, sizes, spot empty chunks, read any chunk in full.

In [1]:
import os
from pathlib import Path
import pandas as pd

def find_backend() -> Path:
    for p in [Path(os.getcwd()).resolve()] + list(Path(os.getcwd()).resolve().parents):
        if (p / 'parent_docstore').exists() and (p / 'data').exists():
            return p
        if (p / 'backend' / 'parent_docstore').exists():
            return p / 'backend'
    raise RuntimeError('Cannot find backend root.')

BACKEND           = find_backend()
PARENT_STORE_PATH = BACKEND / 'parent_docstore'
print(f'✅  Backend: {BACKEND}')

✅  Backend: C:\Users\karth\nexora-sentiobot\backend


In [2]:
from langchain.storage import LocalFileStore
from langchain.storage._lc_store import create_kv_docstore

byte_store = LocalFileStore(str(PARENT_STORE_PATH))
docstore   = create_kv_docstore(byte_store)
all_ids    = list(byte_store.yield_keys())
all_docs   = docstore.mget(all_ids)

rows = []
for doc_id, doc in zip(all_ids, all_docs):
    if doc is None:
        rows.append({'doc_id': doc_id[:16], 'source': 'MISSING', 'section': 'MISSING',
                     'words': 0, 'chars': 0, 'preview': ''})
        continue
    section = (doc.metadata.get('section_title')
               or doc.metadata.get('Category')
               or doc.metadata.get('source', '?'))
    rows.append({
        'doc_id':  doc_id[:16] + '…',
        'source':  doc.metadata.get('source', '?'),
        'section': str(section)[:50],
        'words':   len(doc.page_content.split()),
        'chars':   len(doc.page_content),
        'preview': doc.page_content[:100].replace('\n', ' '),
    })

df = pd.DataFrame(rows)
print(f'Total parent chunks: {len(df)}')
df[['source','section','words','chars']].head(15)

Total parent chunks: 85


,source,section,words,chars
0,policies.md,5. End-of-Life (EOL) Policy,69,442
1,faqs.csv,Lighting,28,156
2,faqs.csv,App,24,161
3,faqs.csv,Thermostat,24,128
4,faqs.csv,Camera,21,137
5,faqs.csv,Lighting,23,134
6,nexora_thermostat_pro_manual.md,7. In-Depth Troubleshooting,344,2357
7,visionsphere_360_manual.md,4. Installation & Initial Setup,269,1626
8,faqs.csv,Returns,26,157
9,faqs.csv,Lighting,26,133


In [3]:
# Chunk count + size stats per source file
print(df.groupby('source').agg(
    chunks=('doc_id','count'),
    avg_words=('words','mean'),
    min_words=('words','min'),
    max_words=('words','max'),
    total_words=('words','sum'),
).round(1).to_string())
print(f'\nGrand total words: {df["words"].sum():,}')

                                   chunks  avg_words  min_words  max_words  total_words
source                                                                                 
faqs.csv                               50       23.4         18         29         1168
lumiglow_smart_lighting_manual.md       9      142.3         10        413         1281
nexora_thermostat_pro_manual.md        10      126.1          9        344         1261
policies.md                             7      132.1         44        289          925
visionsphere_360_manual.md              9      147.9         10        449         1331

Grand total words: 5,966


In [4]:
# Flag tiny chunks
tiny = df[df['words'] < 20]
if tiny.empty:
    print('✅  No tiny chunks (all >= 20 words)')
else:
    print(f'⚠️  {len(tiny)} chunks under 20 words:')
    print(tiny[['source','section','words','preview']].to_string())

⚠️  8 chunks under 20 words:
                               source                                       section  words                                                                                               preview
13    nexora_thermostat_pro_manual.md           Nexora Thermostat Pro – User Manual      9                                       # Nexora Thermostat Pro – User Manual **Model: NTS-PRO-24V-V3**
22                           faqs.csv                                  Connectivity     18  Question: Do Nexora devices support 5GHz WiFi? Answer: Currently Nexora devices support 2.4GHz WiFi 
29                           faqs.csv                                   Integration     19  Question: Can I use voice assistants with the Thermostat Pro? Answer: Yes, it supports Alexa, Google
49                           faqs.csv                                        Camera     18  Question: Can I view live feeds remotely? Answer: Yes, via the Nexora app with secure end-to-end enc
65  lu

In [5]:
# Read any chunk in full — change these filters
SOURCE  = 'policies.md'
SECTION = 'Warranty'

mask = (
    df['source'].str.contains(SOURCE, case=False) &
    df['section'].str.contains(SECTION, case=False)
)
matches = df[mask]
print(f'Found {len(matches)} chunk(s)\n')
for pos in matches.index:
    doc = all_docs[pos]
    print(f"Source  : {df.loc[pos,'source']}")
    print(f"Section : {df.loc[pos,'section']}")
    print(f"Words   : {df.loc[pos,'words']}")
    print()
    print(doc.page_content)
    print('═'*60)

Found 1 chunk(s)

Source  : policies.md
Section : 1. Limited Warranty Policy
Words   : 289

## 1. Limited Warranty Policy
### 1.1. Warranty Period
The warranty period is effective from the original date of purchase and varies by product. The definitive warranty period for your specific device is stated in its user manual. For reference, the standard periods are:

* **Nexora Thermostat Pro:** 2-Year Limited Warranty
* **SecureSphere 360 Camera:** 1-Year Limited Warranty
* **LumiGlow Smart Lights:** 1-Year Limited Warranty
* **All other products & accessories:** Please consult the product's packaging or user manual.

### 1.2. Scope of Coverage
This warranty covers:
- Hardware failures resulting from defects in materials and workmanship under normal use conditions as outlined in the product's user manual.
- Firmware malfunctions that are not resolved by a user-initiated update performed according to Nexora's official instructions.

### 1.3. Exclusions from Coverage
This limited warranty d